# Building a Deep Agent on Nemotron 3 Ultra with DeepInfra

*A hands-on guide: from zero to a running LangChain Deep Agent powered by NVIDIA Nemotron 3 Ultra, hosted on DeepInfra.*

## What you'll build

By the end of this guide you'll have a working Deep Agent — an agent that can plan, use tools, and complete multi-step tasks — running on **Nemotron 3 Ultra** via DeepInfra's OpenAI-compatible API, using LangChain's Deep Agents harness profile tuned for NVIDIA Nemotron 3 Ultra.

## Table of Contents

- [Prerequisites](#prerequisites)
- [Step 1 — Install dependencies](#step-1)
- [Step 2 — Set your DeepInfra token](#step-2)
- [Step 3 — Point a model at DeepInfra](#step-3)
- [Step 4 — Define a tool](#step-4)
- [Step 5 — Create the Deep Agent](#step-5)
- [Step 6 — Run it](#step-6)
- [Best practices](#best-practices)
- [Next steps](#next-steps)

<a id="prerequisites"></a>
## Prerequisites

- A DeepInfra account and API token (get one at [deepinfra.com](https://deepinfra.com))
- Python 3.10 or newer
- Basic familiarity with LangChain (helpful but not required)

<a id="step-1"></a>
## Step 1 — Install dependencies

Install the Deep Agents harness along with LangChain and the OpenAI-compatible chat integration.

In [ ]:
!pip install deepagents langchain langchain-openai

<a id="step-2"></a>
## Step 2 — Set your DeepInfra token

Provide your DeepInfra API token. This cell reads it securely and stores it in the `DEEPINFRA_TOKEN` environment variable for the rest of the notebook.

> Outside a notebook you'd typically export it in your shell instead:
> ```bash
> export DEEPINFRA_TOKEN="your-token-here"
> ```

In [ ]:
import os
import getpass

if not os.environ.get("DEEPINFRA_TOKEN"):
    os.environ["DEEPINFRA_TOKEN"] = getpass.getpass("Enter your DeepInfra API token: ")

<a id="step-3"></a>
## Step 3 — Point a model at DeepInfra

DeepInfra is OpenAI-compatible, so you use `ChatOpenAI` and just swap the base URL and key to point at Nemotron 3 Ultra:

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B",
    base_url="https://api.deepinfra.com/v1/openai",
    api_key=os.environ["DEEPINFRA_TOKEN"],
)

<a id="step-4"></a>
## Step 4 — Define a tool

Agents are only as useful as the tools they can call. Here's a simple example tool — swap in your own logic later:

In [ ]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

<a id="step-5"></a>
## Step 5 — Create the Deep Agent

Deep Agents ship with the tuned Nemotron 3 Ultra harness profile, automatic context compression, a virtual filesystem, and subagent spawning — batteries included. Create one with your model and tools:

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

<a id="step-6"></a>
## Step 6 — Run it

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user",
     "content": "what is the weather in sf"}]}
)

for msg in result["messages"]:
    msg.pretty_print()

That's a complete, working Deep Agent running on Nemotron 3 Ultra via DeepInfra. The agent will plan the task, call your tool as needed, and return a final answer — all using LangChain's harness tuned for this model.

<a id="best-practices"></a>
## Best practices

**Write clear, specific tool descriptions.** The tuned harness relies on good tool descriptions to route correctly — describe what each tool does and when to use it.

**Use subagents for complex tasks.** Deep Agents can spawn subagents to divide work — useful for research or multi-part tasks that would otherwise overflow context.

**Take advantage of the 1M context window.** Nemotron 3 Ultra supports up to 1M tokens — useful for agents that reason over large documents or long histories.

**Trace with LangSmith.** Set `LANGSMITH_TRACING=true` and your API key to inspect tool calls, state transitions, and latency while you iterate.

**Batch non-urgent work.** For large evaluation or offline agent runs, DeepInfra's Batch API can cut cost by 20% versus real-time. For latency-critical traffic, use the Priority Tier.

<a id="next-steps"></a>
## Next steps

- Explore Nemotron 3 Ultra: [deepinfra.com/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B](https://deepinfra.com/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B)
- LangChain Deep Agents docs: [docs.langchain.com/oss/python/deepagents/overview](https://docs.langchain.com/oss/python/deepagents/overview)
- DeepInfra + LangChain integration: [https://docs.deepinfra.com/integrations/langchain](https://docs.deepinfra.com/integrations/langchain)